# Modelagem e Avaliação de Modelos de Ensemble

### Objetivo

- Treinar modelos de ensemble (Random Forest e Gradient Boosting) e um baseline linear (Regressão Logística) reutilizando a **mesma** metodologia do notebook do MLP (mesmo pré-processador, mesma validação cruzada e mesmas métricas), para que a comparação seja justa
- Aplicar Validação Cruzada
- Tratar o desbalanceamento de classes com as mesmas estratégias (threshold, class weight e SMOTE)
- Analisar o trade-off de custo entre falso positivo e falso negativo
- Registrar os resultados dos experimentos em arquivos CSV/JSON
- Comparar o desempenho dos modelos numa tabela consolidada junto com o MLP e escolher o melhor baseado no contexto
- Exportar resultados e modelo final

### Metas

- ROC-AUC maior ou igual a 0,80
- F1-score maior ou igual a 0,60
- recall maior ou igual a 0,55

## Sumário
- [Importando Bibliotecas](#Importando-Bibliotecas)
- [Pré-processamento](#preprocessamento)
    - [Separando Features e Target](#Separando-Features-e-Target)
    - [Separando dados Numéricos e Categóricos](#num-cat)
- [Construindo os Pipelines](#Construindo-os-Pipelines)
    - [Validação Cruzada](#cv)
    - [Funções de Desempenho](#desempenho)
- [Regressão Logística](#lr)
    - [Baseline](#lr-baseline)
    - [Threshold ajustado](#lr-threshold)
- [Random Forest](#rf)
    - [Baseline](#rf-baseline)
    - [Threshold ajustado](#rf-threshold)
    - [Class weight](#rf-classweight)
    - [SMOTE](#rf-smote)
- [Gradient Boosting](#gb)
    - [Baseline](#gb-baseline)
    - [Threshold ajustado](#gb-threshold)
    - [SMOTE](#gb-smote)
- [Conclusão](#conclusao)
    - [Comparação Consolidada](#comparacao)
    - [Análise de Custo](#custo)
    - [Recomendação](#recomendacao)
    - [Exportação](#exportacao)
        - [Modelo](#Modelo)
        - [Resultados](#Resultados)

<h2 id="Importando-Bibliotecas">Importando bibliotecas</h2>

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import joblib


from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix,
)

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

Salvando RANDOM_STATE para permitir reprodutibilidade:

In [2]:
RANDOM_STATE = 42

PROJECT_ROOT = Path.cwd().resolve()
if "notebooks" in str(PROJECT_ROOT):
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = pd.read_csv(DATA_PATH)

pd.set_option("display.max_columns", None)

<h2 id="preprocessamento">Pré-processamento</h2>

In [3]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


A coluna `TotalCharges` foi lida como texto por conter alguns valores em
branco. Convertemos para número; os valores que não convertem viram `NaN` e serão
tratados pelo imputador do pipeline.

In [5]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


Verificamos que os dados estão com os tipos corretos. Antes de separar as
**features** do **target**, olhamos o balanceamento das classes:

In [7]:
balanceamento_classes = pd.DataFrame({
    "Quantidade": df["Churn"].value_counts(),
    "Percentual": df["Churn"].value_counts(normalize=True).mul(100).round(2),
})

balanceamento_classes

,Quantidade,Percentual
Churn,,
No,5174,73.46
Yes,1869,26.54


Como existe um desbalanceamento (cerca de 27% de churn), testamos algumas
estratégias para melhorar o desempenho do modelo levando em consideração a
necessidade do negócio.

### Separando Features e Target

In [8]:
X = df.drop(columns=["customerID", "Churn"])
y = (df["Churn"] == "Yes").astype(int)

<h3 id="num-cat">Separando dados Numéricos e Categóricos</h3>

In [9]:
cols_numericas = ["tenure", "MonthlyCharges", "TotalCharges"]
cols_categoricas = [col for col in X.columns if col not in cols_numericas]

<h2 id="Construindo-os-Pipelines">Construindo os Pipelines</h2>

O `preprocessador` é idêntico ao do notebook do MLP. Modelos de árvore não
exigem escalonamento, mas mantemos o `StandardScaler` para preservar exatamente a
mesma matriz de features e tornar a comparação justa entre os notebooks. Também
já instanciamos os três estimadores que serão avaliados.

In [10]:
pipeline_numerico = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

pipeline_categorico = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessador = ColumnTransformer([
    ("num", pipeline_numerico, cols_numericas),
    ("cat", pipeline_categorico, cols_categoricas),
])

lr = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)

rf = RandomForestClassifier(
    n_estimators=300,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

gb = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    random_state=RANDOM_STATE,
)

<h3 id="cv">Validação Cruzada</h3>

Mesma divisão estratificada em 5 folds usada no notebook do MLP.

In [11]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

<h3 id="desempenho">Funções de Desempenho</h3>

Para não repetir o mesmo código ao longo do treinamento dos modelos, reaproveitamos
as funções do notebook do MLP:
- **calcular_metricas** retorna as métricas para avaliarmos os modelos;
- **cv_resultados** faz a validação cruzada considerando thresholds diversos caso
seja preciso (por padrão o limiar é 0.5). Aqui ela também acumula a **matriz de
confusão** somada entre os folds, que usaremos na análise de custo.

In [12]:
def calcular_metricas(y_true, y_pred, y_proba):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_proba),
    }

In [13]:
def cv_resultados(pipeline, threshold=0.5):
    metricas = []
    matrizes = []
    for train_idx, test_idx in cv.split(X, y):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        pipeline.fit(X_train, y_train)
        proba = pipeline.predict_proba(X_test)[:, 1]
        preds = (proba >= threshold).astype(int)

        metricas.append(calcular_metricas(y_test, preds, proba))
        matrizes.append(confusion_matrix(y_test, preds, labels=[0, 1]))

    media_metricas = pd.DataFrame(metricas).mean()
    media_metricas["threshold"] = threshold
    cm_total = np.sum(matrizes, axis=0)
    return media_metricas, cm_total

As funções abaixo rodam a validação cruzada de cada estratégia, guardam o
resultado para a tabela comparativa final e a matriz de confusão para a análise de
custo. Os resultados são vistos na tabela consolidada, ao final — sem poluir a
saída a cada estratégia (mesma lógica do notebook do MLP).

In [14]:
# acumuladores usados ao longo do notebook
resultados = []
matrizes_confusao = {}

def avaliar(nome, modelo_tipo, pipeline, threshold=0.5):
    """Roda a CV e guarda o resultado e a matriz de confusão da estratégia."""
    media, cm = cv_resultados(pipeline, threshold=threshold)
    linha = media.to_dict()
    linha["estrategia"] = nome
    linha["modelo"] = modelo_tipo
    matrizes_confusao[nome] = cm
    resultados.append(linha)
    return media

<h2 id="lr">Regressão Logística</h2>

Baseline **linear**, com `class_weight="balanced"` para lidar com o
desbalanceamento. É a referência linear que a Etapa 2 pede para comparar contra o
MLP e os modelos de árvore.

<h3 id="lr-baseline">Baseline</h3>

In [15]:
lr_pipeline = Pipeline([
    ("preprocessador", preprocessador),
    ("classificador", lr),
])

_ = avaliar("lr_baseline", "LogisticRegression", lr_pipeline)

<h3 id="lr-threshold">Threshold ajustado</h3>

In [16]:
for thr in [0.30, 0.35, 0.40, 0.45]:
    _ = avaliar(f"lr_threshold_{thr:.2f}", "LogisticRegression", lr_pipeline,
                  threshold=thr)

<h2 id="rf">Random Forest</h2>

Primeiro ensemble: floresta aleatória. Testamos a versão padrão, o ajuste de
threshold, o uso de `class_weight="balanced"` (alternativa nativa das árvores) e
SMOTE.

<h3 id="rf-baseline">Baseline</h3>

In [17]:
rf_pipeline = Pipeline([
    ("preprocessador", preprocessador),
    ("classificador", rf),
])

_ = avaliar("rf_baseline", "RandomForest", rf_pipeline)

<h3 id="rf-threshold">Threshold ajustado</h3>

Reduzir o limiar aumenta o recall — o que interessa em churn — ao custo de
precisão. Testamos a mesma faixa usada no MLP.

In [18]:
for thr in [0.30, 0.35, 0.40, 0.45]:
    _ = avaliar(f"rf_threshold_{thr:.2f}", "RandomForest", rf_pipeline,
                  threshold=thr)

<h3 id="rf-classweight">Class weight</h3>

In [19]:
rf_balanced = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

rf_balanced_pipeline = Pipeline([
    ("preprocessador", preprocessador),
    ("classificador", rf_balanced),
])

_ = avaliar("rf_class_weight", "RandomForest", rf_balanced_pipeline)

<h3 id="rf-smote">SMOTE</h3>

In [20]:
rf_smote_pipeline = ImbPipeline([
    ("preprocessador", preprocessador),
    ("sampler", SMOTE(random_state=RANDOM_STATE)),
    ("classificador", rf),
])

_ = avaliar("rf_smote", "RandomForest", rf_smote_pipeline)

<h2 id="gb">Gradient Boosting</h2>

Segundo ensemble: boosting sequencial de árvores. O `GradientBoostingClassifier`
não tem `class_weight`, então tratamos o desbalanceamento por threshold e SMOTE.

<h3 id="gb-baseline">Baseline</h3>

In [21]:
gb_pipeline = Pipeline([
    ("preprocessador", preprocessador),
    ("classificador", gb),
])

_ = avaliar("gb_baseline", "GradientBoosting", gb_pipeline)

<h3 id="gb-threshold">Threshold ajustado</h3>

In [22]:
for thr in [0.30, 0.35, 0.40, 0.45]:
    _ = avaliar(f"gb_threshold_{thr:.2f}", "GradientBoosting", gb_pipeline,
                  threshold=thr)

<h3 id="gb-smote">SMOTE</h3>

In [23]:
gb_smote_pipeline = ImbPipeline([
    ("preprocessador", preprocessador),
    ("sampler", SMOTE(random_state=RANDOM_STATE)),
    ("classificador", gb),
])

_ = avaliar("gb_smote", "GradientBoosting", gb_smote_pipeline)

<h2 id="conclusao">Conclusão</h2>

Agora, vamos comparar os resultados obtidos nas tabelas abaixo.

<h3 id="comparacao">Comparação Consolidada</h3>

Montamos a tabela com todos os modelos deste notebook (Regressão Logística, RF e
GB) e carregamos os resultados do MLP (salvos em `reports/mlp_resultados.csv` pelo
notebook anterior) para consolidar tudo num único ranking, ordenado por F1-score.

In [24]:
ensemble_df = pd.DataFrame(resultados)[
    ["modelo", "estrategia", "accuracy", "precision", "recall", "f1", "roc_auc"]
]

mlp_csv = PROJECT_ROOT / "reports" / "mlp_resultados.csv"
if mlp_csv.exists():
    mlp_df = pd.read_csv(mlp_csv)
    mlp_df["estrategia"] = "mlp_" + mlp_df["estrategia"].astype(str)
    mlp_df["modelo"] = "MLP"
    mlp_df = mlp_df[["modelo", "estrategia", "accuracy", "precision", "recall", "f1", "roc_auc"]]
    consolidado = pd.concat([mlp_df, ensemble_df], ignore_index=True)
else:
    print("Aviso: reports/mlp_resultados.csv não encontrado. Mostrando só este notebook.")
    consolidado = ensemble_df.copy()

consolidado = consolidado.sort_values("f1", ascending=False).reset_index(drop=True)
consolidado

,modelo,estrategia,accuracy,precision,recall,f1,roc_auc
0,GradientBoosting,gb_threshold_0.35,0.782618,0.574675,0.696082,0.629562,0.845906
1,GradientBoosting,gb_threshold_0.30,0.765297,0.541780,0.750135,0.629107,0.845906
2,MLP,mlp_undersampling,0.753509,0.524748,0.775805,0.625835,0.841632
3,LogisticRegression,lr_baseline,0.745278,0.512834,0.802019,0.625590,0.844865
4,MLP,mlp_threshold_0.35,0.777081,0.565149,0.697649,0.623464,0.843306
5,MLP,mlp_smote,0.745277,0.513035,0.790786,0.622159,0.842209
6,GradientBoosting,gb_smote,0.789433,0.593759,0.652735,0.621765,0.844067
7,GradientBoosting,gb_threshold_0.40,0.792557,0.602509,0.641505,0.621347,0.845906
8,MLP,mlp_threshold_0.30,0.758482,0.532224,0.745809,0.620508,0.843306
9,MLP,mlp_threshold_0.40,0.788155,0.592015,0.651105,0.619265,0.843306


<h3 id="custo">Análise de Custo (FP × FN)</h3>

Em churn, os dois tipos de erro têm custos diferentes:

- **Falso negativo (FN)**: o modelo diz que o cliente fica, mas ele cancela.
  Perde-se o cliente sem chance de retê-lo — custo alto (adquirir um novo cliente
  costuma ser bem mais caro que reter um existente).
- **Falso positivo (FP)**: o modelo diz que o cliente vai sair, mas ele ficaria.
  Gasta-se uma ação de retenção (desconto, contato) desnecessária — custo menor.

Assumimos, como premissa de negócio a validar com stakeholders, que **um FN custa
5× mais que um FP**. Com as matrizes de confusão acumuladas na validação cruzada
(portanto, sobre todo o dataset), calculamos o custo total de cada estratégia e
ordenamos da menor para a maior.

In [25]:
CUSTO_FN = 5   # perder um cliente que ia cancelar
CUSTO_FP = 1   # ação de retenção desnecessária

linhas_custo = []
for nome, cm in matrizes_confusao.items():
    tn, fp, fn, tp = cm.ravel()
    custo = fn * CUSTO_FN + fp * CUSTO_FP
    linhas_custo.append({
        "estrategia": nome,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "custo_total": int(custo),
    })

custo_df = pd.DataFrame(linhas_custo).sort_values("custo_total").reset_index(drop=True)
custo_df

,estrategia,tn,fp,fn,tp,custo_total
0,lr_threshold_0.35,3042,2132,169,1700,2977
1,lr_threshold_0.30,2825,2349,142,1727,3059
2,lr_threshold_0.40,3324,1850,244,1625,3070
3,lr_threshold_0.45,3543,1631,306,1563,3161
4,lr_baseline,3750,1424,370,1499,3274
5,gb_threshold_0.30,3988,1186,467,1402,3521
6,gb_threshold_0.35,4211,963,568,1301,3803
7,rf_threshold_0.30,3920,1254,510,1359,3804
8,gb_smote,4340,834,649,1220,4079
9,rf_threshold_0.35,4124,1050,614,1255,4120


<h3 id="recomendacao">Recomendação</h3>

O modelo de ensemble com melhor desempenho foi o **Gradient Boosting com
threshold ajustado para 0,35**, que superou o MLP campeão da Etapa 2 justamente
nas métricas que mais importam para churn: obteve maior recall (0,696 contra 0,651)
e maior F1-score (0,630 contra 0,619), com ROC-AUC praticamente igual (0,846 contra
0,843). Em troca, teve acurácia e precisão ligeiramente menores, o que é aceitável:
como o objetivo é identificar clientes propensos ao cancelamento, capturar mais
churners reais (recall) vale mais do que maximizar a acurácia global.

A análise de custo (FP × FN, assumindo que um falso negativo custa 5× um falso
positivo) reforça a escolha. Estratégias de threshold muito baixo capturam quase
todos os churners, mas geram um volume alto de falsos positivos — o que na prática
significaria acionar retenção para muita gente que não iria cancelar. O Gradient
Boosting com threshold 0,35 equilibra recall alto com um número de falsos positivos
mais gerenciável. Vale notar que essa premissa de custo (5:1) deve ser validada com
os stakeholders; a decisão final entre "capturar o máximo de churners" e "não
desperdiçar ações de retenção" depende da capacidade real da operação de retenção.

Diante do equilíbrio entre F1, recall e um volume gerenciável de falsos positivos,
o **Gradient Boosting com threshold 0,35** foi escolhido como modelo campeão do
ensemble e exportado como artefato final.

<h3 id="exportacao">Exportação</h3>

#### Modelo

Treinando a pipeline campeã com todos os dados e exportando o artefato no mesmo
formato do notebook do MLP (dicionário com pipeline, threshold e nome do modelo),
para manter a compatibilidade com a etapa de API.

In [26]:
# ajuste para a estratégia vencedora escolhida na recomendação acima
ESTRATEGIA_CAMPEA = ensemble_df.sort_values("f1", ascending=False).iloc[0]["estrategia"]

def resolver_pipeline(nome):
    thr = 0.5
    if "threshold" in nome:
        thr = float(nome.split("_")[-1])
    if nome.startswith("lr"):
        return lr_pipeline, thr
    if nome.startswith("rf_class_weight"):
        return rf_balanced_pipeline, thr
    if nome.startswith("rf_smote"):
        return rf_smote_pipeline, thr
    if nome.startswith("rf"):
        return rf_pipeline, thr
    if nome.startswith("gb_smote"):
        return gb_smote_pipeline, thr
    if nome.startswith("gb"):
        return gb_pipeline, thr
    raise ValueError(f"Estratégia desconhecida: {nome}")

pipeline_campea, threshold_campeao = resolver_pipeline(ESTRATEGIA_CAMPEA)
pipeline_campea.fit(X, y)

artefato = {
    "pipeline": pipeline_campea,
    "threshold": threshold_campeao,
    "model_name": ESTRATEGIA_CAMPEA,
}

(PROJECT_ROOT / "models").mkdir(parents=True, exist_ok=True)
joblib.dump(
    artefato,
    PROJECT_ROOT / "models" / f"ensemble_{ESTRATEGIA_CAMPEA}.joblib"
)

print("Modelo exportado!")

Modelo exportado!


#### Resultados

Salvando os resultados de todos os modelos, a tabela consolidada e a análise de
custo em `reports/`, e as métricas do campeão em `reports/metrics/`, seguindo o
mesmo padrão do notebook do MLP.

In [27]:
(PROJECT_ROOT / "reports" / "metrics").mkdir(parents=True, exist_ok=True)

ensemble_df.to_csv(PROJECT_ROOT / "reports" / "ensemble_resultados.csv", index=False)
consolidado.to_csv(PROJECT_ROOT / "reports" / "comparativo_consolidado.csv", index=False)
custo_df.to_csv(PROJECT_ROOT / "reports" / "ensemble_custo.csv", index=False)

print("Resultados salvos!")

Resultados salvos!


Salvando resultados do modelo campeão

In [28]:
metricas_campea = ensemble_df[ensemble_df["estrategia"] == ESTRATEGIA_CAMPEA][
    ["accuracy", "precision", "recall", "f1", "roc_auc"]
]

metricas_campea.to_json(
    PROJECT_ROOT / "reports" / "metrics" / f"metricas_ensemble_{ESTRATEGIA_CAMPEA}.json",
    orient="records",
    indent=2,
)

print("Resultados salvos!")

Resultados salvos!
